In [1]:
import os
import copy
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path

import torch
import timm
import torchvision.transforms as transforms

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import (
    StratifiedKFold, cross_val_score,
    RandomizedSearchCV, train_test_split
)
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.ensemble import VotingClassifier
from scipy.stats import loguniform
from xgboost import XGBClassifier

# GPU-accelerated models (cuML) with sklearn fallback
try:
    from cuml.linear_model import LogisticRegression
    from cuml.svm import SVC
    print("Using cuML (GPU) for Logistic Regression and SVM")
except ImportError:
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import SVC
    print("cuML not found, falling back to sklearn (CPU)")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Torch device: {DEVICE}")

c:\Users\MCuartero\Uni\machine-learning\fine-grained-visual-recognition\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuML not found, falling back to sklearn (CPU)
Torch device: cpu


In [2]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    print("Running on Google Colab. Setting up Drive paths...")

    drive.mount("/content/drive")

    ZIP_PATH = Path("/content/drive/MyDrive/animals.zip")
    EXTRACT_DIR = Path("/content/data")

    checkpoint_1 = EXTRACT_DIR / "train_metadata.csv"
    checkpoint_2 = EXTRACT_DIR / "animals" / "train_metadata.csv"

    if checkpoint_1.exists() or checkpoint_2.exists():
        print("Data already unzipped.")
    else:
        if ZIP_PATH.exists():
            print("Unzipping now...")
            os.system(f"unzip -q {ZIP_PATH} -d {EXTRACT_DIR}")
            print("Unzipping complete!")
        else:
            print("Could not find 'animals.zip' in the root of your Google Drive.")
else:
    print("Running locally. Skipping Google Drive & unzip setup.")

Running locally. Skipping Google Drive & unzip setup.


In [3]:
IN_COLAB = "google.colab" in sys.modules
HAS_CUDA = torch.cuda.is_available()

if HAS_CUDA:
    BATCH_SIZE = 64
    XGB_TREES = 300
    SVM_ITER = 15
elif IN_COLAB:
    BATCH_SIZE = 32
    XGB_TREES = 200
    SVM_ITER = 10
else:
    BATCH_SIZE = 32
    XGB_TREES = 100
    SVM_ITER = 5

if IN_COLAB:
    EXTRACT_DIR = Path("/content/data")
    BASE_DIR = (EXTRACT_DIR / "animals") if (EXTRACT_DIR / "animals").exists() else EXTRACT_DIR
else:
    BASE_DIR = Path("data/animals")

TRAIN_META = BASE_DIR / "train_metadata.csv"
TEST_META = BASE_DIR / "test_metadata.csv"
COLOR_HIST = BASE_DIR / "color_histogram.csv"
HOG_PCA = BASE_DIR / "hog_pca.csv"
ADDITIONAL_FEATS = BASE_DIR / "additional_features.csv"

TRAIN_VIT_CACHE = BASE_DIR / "train_vit_base_embeddings.npy"
TEST_VIT_CACHE = BASE_DIR / "test_vit_base_embeddings.npy"
TRAIN_RN_CACHE = BASE_DIR / "train_resnet50_embeddings.npy"
TEST_RN_CACHE = BASE_DIR / "test_resnet50_embeddings.npy"

SUBMISSION_PATH = BASE_DIR / "submission.csv"

print(f"Using device: {DEVICE}")

Using device: cpu


In [4]:
print("\n--- Loading CSV features ---")

train_meta = pd.read_csv(TRAIN_META)
test_meta = pd.read_csv(TEST_META)

color = pd.read_csv(COLOR_HIST)
hog = pd.read_csv(HOG_PCA)
add = pd.read_csv(ADDITIONAL_FEATS)

def merge_features(meta, *feature_dfs):
    df = meta.copy()
    for fdf in feature_dfs:
        df = df.merge(fdf, on="image_id", how="left")
    return df

train_df = merge_features(train_meta, color, hog, add)
test_df = merge_features(test_meta, color, hog, add)

meta_cols = ["image_id", "image_path", "class_id", "class_name"]
csv_feature_cols = [c for c in train_df.columns if c not in meta_cols]

print(f"Train samples: {len(train_df)},  Test samples: {len(test_df)}")
print(f"CSV feature dims: {len(csv_feature_cols)}")


--- Loading CSV features ---
Train samples: 3750,  Test samples: 1250
CSV feature dims: 219


In [5]:
def extract_embeddings(image_paths, base_dir, cache_path, model_name, batch_size=32):
    if cache_path.exists():
        print(f"  Loading cached embeddings from {cache_path}")
        return np.load(cache_path)

    print(f"  Extracting {model_name} embeddings for {len(image_paths)} images ...")

    model = timm.create_model(model_name, pretrained=True, num_classes=0)
    model.eval().to(DEVICE)

    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    all_embeddings = []
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i: i + batch_size]
        imgs = []
        for rel_path in batch_paths:
            full_path = base_dir / rel_path
            try:
                img = Image.open(full_path).convert("RGB")
            except Exception:
                img = Image.new("RGB", (64, 64), (128, 128, 128))
            imgs.append(preprocess(img))

        batch_tensor = torch.stack(imgs).to(DEVICE)
        with torch.no_grad():
            emb = model(batch_tensor).cpu().numpy()
        all_embeddings.append(emb)

        if (i // batch_size) % 10 == 0:
            print(f"    {min(i + batch_size, len(image_paths))}/{len(image_paths)}")

    embeddings = np.vstack(all_embeddings)
    np.save(cache_path, embeddings)
    print(f"  Saved to {cache_path}")
    return embeddings


print("\n--- Extracting ViT-Base embeddings ---")
train_vit = extract_embeddings(
    train_df["image_path"].values, BASE_DIR,
    TRAIN_VIT_CACHE, "vit_base_patch16_224", BATCH_SIZE
)
test_vit = extract_embeddings(
    test_df["image_path"].values, BASE_DIR,
    TEST_VIT_CACHE, "vit_base_patch16_224", BATCH_SIZE
)

print("\n--- Extracting ResNet-50 embeddings ---")
train_rn = extract_embeddings(
    train_df["image_path"].values, BASE_DIR,
    TRAIN_RN_CACHE, "resnet50", BATCH_SIZE
)
test_rn = extract_embeddings(
    test_df["image_path"].values, BASE_DIR,
    TEST_RN_CACHE, "resnet50", BATCH_SIZE
)

print(f"ViT-Base shape:  {train_vit.shape}")
print(f"ResNet-50 shape: {train_rn.shape}")


--- Extracting ViT-Base embeddings ---
  Loading cached embeddings from data\animals\train_vit_base_embeddings.npy
  Loading cached embeddings from data\animals\test_vit_base_embeddings.npy

--- Extracting ResNet-50 embeddings ---
  Loading cached embeddings from data\animals\train_resnet50_embeddings.npy
  Loading cached embeddings from data\animals\test_resnet50_embeddings.npy
ViT-Base shape:  (3750, 768)
ResNet-50 shape: (3750, 2048)


In [ ]:
print("\n--- Combining and normalising features ---")

X_train_csv = np.nan_to_num(train_df[csv_feature_cols].values.astype(np.float32))
X_test_csv = np.nan_to_num(test_df[csv_feature_cols].values.astype(np.float32))
scaler_csv = StandardScaler()
X_train_csv = scaler_csv.fit_transform(X_train_csv)
X_test_csv = scaler_csv.transform(X_test_csv)

scaler_vit = StandardScaler()
X_train_vit = scaler_vit.fit_transform(train_vit)
X_test_vit = scaler_vit.transform(test_vit)

scaler_rn = StandardScaler()
X_train_rn = scaler_rn.fit_transform(train_rn)
X_test_rn = scaler_rn.transform(test_rn)

# PCA to reduce dimensionality before concatenation
pca_vit = PCA(n_components=128, random_state=42)
X_train_vit = pca_vit.fit_transform(X_train_vit)
X_test_vit = pca_vit.transform(X_test_vit)

pca_rn = PCA(n_components=128, random_state=42)
X_train_rn = pca_rn.fit_transform(X_train_rn)
X_test_rn = pca_rn.transform(X_test_rn)

X_train = np.hstack([X_train_vit, X_train_rn, X_train_csv])
X_test = np.hstack([X_test_vit, X_test_rn, X_test_csv])

y_train = train_df["class_id"].values

print(f"Total feature dims: {X_train.shape[1]}")
print(f"  ViT-Base (PCA): {X_train_vit.shape[1]}")
print(f"  ResNet-50 (PCA): {X_train_rn.shape[1]}")
print(f"  CSV: {X_train_csv.shape[1]}")


--- Combining and normalising features ---
Total feature dims: 475
  ViT-Base (PCA):  128
  ResNet-50 (PCA): 128
  CSV:             219


In [7]:
def cv_evaluate(model, X, y, n_splits=5, label="Model"):
    """Stratified 5-fold cross-validation."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=skf, scoring="accuracy", n_jobs=-1)
    print(f"  {label}: {scores.mean():.4f} ± {scores.std():.4f}  (folds: {np.round(scores, 4)})")
    return scores.mean()

In [8]:
print("\n--- Model 1: Logistic Regression ---")

is_cuml = LogisticRegression.__module__.startswith("cuml")
solver_name = "qn" if is_cuml else "lbfgs"

lr = LogisticRegression(
    C=10.0,
    max_iter=2000,
    solver=solver_name,
    class_weight="balanced"
)

lr_cv = cv_evaluate(lr, X_train, y_train, label="Logistic Regression")


--- Model 1: Logistic Regression ---
  Logistic Regression: 0.9389 ± 0.0120  (folds: [0.9213 0.9413 0.9573 0.9427 0.932 ])


In [9]:
print("\n--- Model 2: SVM (RBF) with RandomizedSearchCV ---")

is_cuml = SVC.__module__.startswith("cuml")

if is_cuml:
    svc_kwargs = {"kernel": "rbf", "probability": True}
else:
    svc_kwargs = {"kernel": "rbf", "probability": False, "random_state": 42}

param_dist = {
    "C": loguniform(1, 1000),
    "gamma": loguniform(1e-4, 1e-1),
}

svm_search = RandomizedSearchCV(
    SVC(**svc_kwargs),
    param_distributions=param_dist,
    n_iter=SVM_ITER,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    random_state=42,
    verbose=3
)

print(f"Starting RandomizedSearchCV ({SVM_ITER} iterations, 3 folds)...")
svm_search.fit(X_train, y_train)

print(f"  Best params: {svm_search.best_params_}")
print(f"  Best CV score: {svm_search.best_score_:.4f}")

if not is_cuml:
    svm = SVC(
        kernel="rbf",
        probability=True,
        random_state=42,
        **svm_search.best_params_
    )
    svm.fit(X_train, y_train)
else:
    svm = svm_search.best_estimator_

svm_cv = svm_search.best_score_


--- Model 2: SVM (RBF) with RandomizedSearchCV ---
Starting RandomizedSearchCV (5 iterations, 3 folds)...
Fitting 3 folds for each of 5 candidates, totalling 15 fits
  Best params: {'C': np.float64(2.9380279387035344), 'gamma': np.float64(0.00029375384576328325)}
  Best CV score: 0.9507


In [10]:
print("\n--- Model 3: XGBoost ---")

xgb = XGBClassifier(
    n_estimators=XGB_TREES,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.6,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    device=DEVICE
)

xgb_cv = cv_evaluate(xgb, X_train, y_train, label="XGBoost")


--- Model 3: XGBoost ---
  XGBoost: 0.9341 ± 0.0142  (folds: [0.912  0.944  0.9453 0.9467 0.9227])


In [11]:
print("\n--- Model 4: Soft Voting Ensemble ---")

is_cuml = LogisticRegression.__module__.startswith("cuml")
best_C = svm_search.best_params_["C"]
best_gamma = svm_search.best_params_["gamma"]

if is_cuml:
    lr_inst = LogisticRegression(C=10.0, max_iter=2000, solver="qn", class_weight="balanced")
    svm_inst = SVC(kernel="rbf", probability=True, C=best_C, gamma=best_gamma)
else:
    lr_inst = LogisticRegression(C=10.0, max_iter=2000, solver="lbfgs", class_weight="balanced")
    svm_inst = SVC(kernel="rbf", probability=True, C=best_C, gamma=best_gamma, random_state=42)

ensemble = VotingClassifier(
    estimators=[
        ("lr", lr_inst),
        ("svm", svm_inst),
        ("xgb", XGBClassifier(
            n_estimators=XGB_TREES,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.6,
            min_child_weight=3,
            reg_alpha=0.1,
            reg_lambda=1.0,
            eval_metric="mlogloss",
            random_state=42,
            n_jobs=-1,
            device=DEVICE
        )),
    ],
    voting="soft"
)

ensemble_cv = cv_evaluate(ensemble, X_train, y_train, label="Soft Voting Ensemble")


--- Model 4: Soft Voting Ensemble ---
  Soft Voting Ensemble: 0.9501 ± 0.0090  (folds: [0.9373 0.956  0.96   0.956  0.9413])


In [12]:
print("\n--- Cross-validation summary ---")

results = {
    "Logistic Regression": lr_cv,
    "SVM RBF (tuned)": svm_cv,
    "XGBoost": xgb_cv,
    "Soft Voting Ensemble": ensemble_cv,
}
for name, score in sorted(results.items(), key=lambda x: -x[1]):
    print(f"  {name}: {score:.4f}")

best_name = max(results, key=results.get)
model_map = {
    "Logistic Regression": lr,
    "SVM RBF (tuned)": svm,
    "XGBoost": xgb,
    "Soft Voting Ensemble": ensemble,
}
best_model = model_map[best_name]
print(f"\nBest model: {best_name} ({results[best_name]:.4f})")

print(f"Fitting {best_name} on full training set...")
best_model.fit(X_train, y_train)
train_preds = best_model.predict(X_train)
print(f"Training accuracy (sanity check): {accuracy_score(y_train, train_preds):.4f}")


--- Cross-validation summary ---
  SVM RBF (tuned): 0.9507
  Soft Voting Ensemble: 0.9501
  Logistic Regression: 0.9389
  XGBoost: 0.9341

Best model: SVM RBF (tuned) (0.9507)
Fitting SVM RBF (tuned) on full training set...
Training accuracy (sanity check): 0.9989


In [ ]:
print("\n--- Generating confusion matrix on 20% holdout ---")

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

eval_model = copy.deepcopy(best_model)
eval_model.fit(X_tr, y_tr)
val_preds = eval_model.predict(X_val)
val_acc = accuracy_score(y_val, val_preds)
print(f"Holdout validation accuracy: {val_acc:.4f}")

class_names = (
    train_df[["class_id", "class_name"]]
    .drop_duplicates()
    .sort_values("class_id")["class_name"]
    .tolist()
)

cm = confusion_matrix(y_val, val_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title(f"Confusion Matrix — {best_name} (val acc: {val_acc:.3f})")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()

output_dir = "figures"
output_dir.mkdir(parents=True, exist_ok=True)
safe_model_name = best_name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
plt.savefig(output_dir / f"{safe_model_name}.png", dpi=150)
plt.show()
print(f"Saved: {output_dir / f'{safe_model_name}.png'}")

print("\nClassification Report:")
print(classification_report(y_val, val_preds, target_names=class_names))


--- Generating confusion matrix on 20% holdout ---
Holdout validation accuracy: 0.9427
Saved: data\animals\figures\svm_rbf_tuned.png

Classification Report:
              precision    recall  f1-score   support

        bird       0.99      0.93      0.96        75
   butterfly       1.00      0.95      0.97        75
         cat       0.82      0.85      0.84        75
        deer       0.91      0.96      0.94        75
         dog       0.90      0.85      0.88        75
    elephant       0.96      0.99      0.97        75
        frog       0.94      0.97      0.95        75
       horse       0.97      0.96      0.97        75
       sheep       1.00      0.96      0.98        75
      spider       0.95      1.00      0.97        75

    accuracy                           0.94       750
   macro avg       0.94      0.94      0.94       750
weighted avg       0.94      0.94      0.94       750



In [14]:
print("\n--- Generating test predictions ---")

test_preds = best_model.predict(X_test)
submission = pd.DataFrame({
    "image_id": test_meta["image_id"],
    "class_id": test_preds,
})
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Saved: {SUBMISSION_PATH}  ({len(submission)} rows)")


--- Generating test predictions ---
Saved: data\animals\submission.csv  (1250 rows)
